# Financial PII Masker

A single class that takes free-text (statements, emails, chat transcripts, support tickets) and can mask financial/personal PII on demand, while still being able to hand back the original text. Nothing is persisted anywhere — it only lives for as long as the object does (session memory only).

**PII categories covered (finance-specific + generic):**
- Person name, phone number, email, address, date of birth / dates
- Credit/debit card number, CVV
- Bank account number, IBAN, IFSC code (India), SWIFT/routing-style codes
- Government IDs: PAN (India), Aadhaar (India), SSN (US), passport number, driver's license
- Customer ID / account reference numbers

Built on Presidio (same library as the other notebooks in this folder) so detection reuses a mature NLP + regex engine instead of hand-rolled parsing.

In [ ]:
# Install Presidio + spaCy English model (skip if already installed in this environment)
!pip install -q presidio_analyzer presidio_anonymizer
!python -m spacy download en_core_web_lg


In [ ]:
import re

from presidio_analyzer import AnalyzerEngine, PatternRecognizer, Pattern
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig


In [ ]:
class FinancialPIIMasker:
    """Wraps one piece of text; mask_data() redacts PII, original_data() returns it unchanged.

    Nothing is written to disk or a database — state lives only on the instance,
    for the lifetime of the Python session.
    """

    # Built-in Presidio entities relevant to finance text
    _BUILTIN_ENTITIES = [
        "PERSON", "PHONE_NUMBER", "EMAIL_ADDRESS", "LOCATION", "DATE_TIME",
        "CREDIT_CARD", "IBAN_CODE", "US_SSN", "US_PASSPORT", "US_DRIVER_LICENSE",
        "US_BANK_NUMBER", "CRYPTO",
    ]

    # (entity_name, regex, score) for things Presidio has no built-in recognizer for.
    # ponytail: regexes are heuristics, not validators (e.g. PAN/Aadhaar don't checksum-verify) —
    # upgrade to a proper checksum recognizer if false positives/negatives start to matter.
    _CUSTOM_PATTERNS = [
        ("PAN_NUMBER", r"\b[A-Z]{5}[0-9]{4}[A-Z]\b", 0.9),
        ("AADHAAR_NUMBER", r"\b\d{4}[ -]?\d{4}[ -]?\d{4}\b", 0.75),
        ("IFSC_CODE", r"\b[A-Z]{4}0[A-Z0-9]{6}\b", 0.9),
        ("CUSTOMER_ID", r"\bCUST-\d{6,10}\b", 0.9),
        ("CVV", r"\bCVV[:\s]*\d{3,4}\b", 0.85),
        ("IVA_CASE_REF", r"\bIVA\w{4,}\b", 0.85),  # UK Individual Voluntary Arrangement case refs
        # "<Name> - In Individual Voluntary Arrangement" title lines (1-5 capitalized
        # words right before the fixed suffix; the suffix itself is left untouched).
        ("IVA_TITLE_NAME", r"[A-Z][A-Za-z'.]+(?:\s+[A-Za-z'.]+){0,4}(?=\s*-\s*In Individual Voluntary Arrangement)", 0.85),
    ]

    # Labels whose *value* must always be fully redacted, regardless of what spaCy's NER
    # thinks. Presidio's PERSON/LOCATION recognizers are statistical NER and missed
    # garbled/unusual names and multi-line UK addresses in real IVA statements (see the
    # dummy1/2/3.pdf leak report) — these fields are structured (label: value), so a
    # label-anchored regex is deterministic where NER wasn't.
    # ponytail: this list is tied to the "Protocol Compliant Annual Report" IVA template
    # the dummy PDFs use. It generalizes to any document using that same layout/labels,
    # not to an unrelated document format — a genuinely different template would need its
    # own labels added here.
    _LABELED_FIELDS = ["Previous Address", "Address", "Debtor", "Supervisor", "Signature of Supervisor"]

    # Labels that mark where a labeled field's value ends (the next field begins).
    # Deliberately excludes "Signature of Supervisor": it's always the last field on the
    # page, and adding it here would risk matching "Supervisor" as a false stop-point too.
    _STOP_LABELS = [
        "Date of Birth", "Number of Owned Properties", "Date Appointed", "Date From",
        "Previous Address", "Address", "Debtor", "Supervisor", "Creditors", "Creditor Name",
        "IP Case Ref", "Date Report Issued", "IP Company",
    ]

    _analyzer = None                  # built once, shared across instances (model loading is slow)
    _labeled_field_patterns = None

    def __init__(self, text: str):
        self._original = text
        self._masked = None
        if FinancialPIIMasker._analyzer is None:
            FinancialPIIMasker._analyzer = self._build_analyzer()
        if FinancialPIIMasker._labeled_field_patterns is None:
            FinancialPIIMasker._labeled_field_patterns = self._build_labeled_field_patterns()
        self._anonymizer = AnonymizerEngine()

    @classmethod
    def _build_analyzer(cls):
        analyzer = AnalyzerEngine()
        for entity, regex, score in cls._CUSTOM_PATTERNS:
            analyzer.registry.add_recognizer(
                PatternRecognizer(
                    supported_entity=entity,
                    patterns=[Pattern(name=f"{entity.lower()}_pattern", regex=regex, score=score)],
                )
            )
        return analyzer

    @classmethod
    def _build_labeled_field_patterns(cls):
        stop_alt = "|".join(re.escape(label) for label in cls._STOP_LABELS)
        patterns = []
        for label in cls._LABELED_FIELDS:
            # "Address" must not also match the "Address:" tail end of "Previous Address:"
            lookbehind = r"(?<!Previous )" if label == "Address" else ""
            # Every label in this template is followed by a colon in the source text,
            # except "Signature of Supervisor" (just a line break). Keeping the colon
            # mandatory elsewhere matters: without it, a debtor whose own name contains a
            # label word (e.g. "Miss Debtor Angelne Dummy") would false-match mid-value.
            colon = r"\s*:?\s*" if label == "Signature of Supervisor" else r"\s*:\s*"
            label_and_sep = rf"{re.escape(label)}(?![A-Za-z]){colon}"
            patterns.append(re.compile(
                rf"{lookbehind}({label_and_sep})(.*?)(?=\s*(?:{stop_alt})\s*:|\n\s*\n|\Z)",
                re.IGNORECASE | re.DOTALL,
            ))
        return patterns

    def _mask_labeled_fields(self, text: str) -> str:
        """Blank out the value of each structured label (Debtor/Address/Supervisor/...)."""
        for pattern in self._labeled_field_patterns:
            text = pattern.sub(lambda m: m.group(1) + "<REDACTED>", text)
        return text

    def mask_data(self) -> str:
        """Return the text with all detected PII redacted."""
        pre_masked = self._mask_labeled_fields(self._original)
        entities = self._BUILTIN_ENTITIES + [p[0] for p in self._CUSTOM_PATTERNS]
        results = self._analyzer.analyze(text=pre_masked, entities=entities, language="en")
        anonymized = self._anonymizer.anonymize(
            text=pre_masked,
            analyzer_results=results,
            operators={"DEFAULT": OperatorConfig("replace", {"new_value": "<REDACTED>"})},
        )
        self._masked = anonymized.text
        return self._masked

    def original_data(self) -> str:
        """Return the untouched original text."""
        return self._original


## Demo (fictional data only)

In [ ]:
sample_text = """
Dear Rahul Menon,

Your HDFC credit card ending in 4111 1111 1111 1111 (CVV: 123) was used on 12-Aug-2026.
Customer ID: CUST-78451236. Account IFSC: HDFC0001234. Aadhaar on file: 1234 5678 9012.
PAN: ABCDE1234F. Contact us at +91 98765 43210 or rahul.menon@example.com.
Address on record: 42 MG Road, Ernakulam, Kochi, Kerala 682016.
"""

masker = FinancialPIIMasker(sample_text)

print("MASKED:\n", masker.mask_data())
print("\nORIGINAL (unchanged, still available):\n", masker.original_data())


## Self-check

Minimal assert-based check: masking must actually remove the sensitive substrings, and `original_data()` must stay untouched no matter how many times `mask_data()` is called.

In [ ]:
def demo():
    text = "Contact Rahul Menon at rahul.menon@example.com, card 4111 1111 1111 1111, PAN ABCDE1234F."
    m = FinancialPIIMasker(text)

    masked = m.mask_data()
    assert "rahul.menon@example.com" not in masked
    assert "4111 1111 1111 1111" not in masked
    assert "ABCDE1234F" not in masked

    # original_data must be unaffected by masking, and stable across repeated calls
    assert m.original_data() == text
    m.mask_data()
    assert m.original_data() == text

    print("All self-checks passed.")

demo()


## Load tests: 10 / 20 / 30 / 50 / 100 page statements

Each test case is two cells:
1. **Generate** — builds a synthetic multi-page financial statement (`WORDS_PER_PAGE = 500`, the standard word-processed-page estimate) with unique, fictional PII injected into every record, and returns the list of injected values.
2. **Check** — runs `mask_data()`, times it (wall clock + CPU time → utilisation %), measures memory (stdlib `tracemalloc`, plus process RSS via `psutil` if installed), and asserts none of the injected regex-detectable PII (email, phone, credit card, PAN, Aadhaar, IFSC, customer ID, CVV) survived masking. `PERSON`/`DATE_TIME`/`LOCATION` hits are NLP-based (spaCy), not regex-guaranteed, so those are reported as counts rather than hard-asserted — flag it if that count looks low relative to injected records.

In [ ]:
import os
import random
import time
import tracemalloc

try:
    import psutil  # optional; only used if already installed
except ImportError:
    psutil = None

WORDS_PER_PAGE = 500  # standard word-processed-page estimate


def generate_financial_text(pages: int, seed: int = 42):
    """Build a synthetic multi-page financial statement.

    Returns (text, injected_pii) where injected_pii is every regex-detectable
    fictional PII value planted in the text (used to check nothing leaked through).
    """
    rng = random.Random(seed)
    first_names = ["Rahul", "Ananya", "Vikram", "Priya", "Arjun", "Divya", "Karan", "Meera"]
    last_names = ["Menon", "Nair", "Iyer", "Reddy", "Singh", "Gupta", "Rao", "Kapoor"]
    cities = ["Kochi", "Chennai", "Bangalore", "Mumbai", "Pune", "Hyderabad", "Delhi", "Jaipur"]

    target_words = pages * WORDS_PER_PAGE
    records = []
    injected = []
    word_count = 0
    i = 0
    while word_count < target_words:
        i += 1
        name = f"{rng.choice(first_names)} {rng.choice(last_names)}"
        email = f"customer{i}@example.com"
        phone = f"+91 9{rng.randint(100000000, 999999999)}"
        card = f"{rng.randint(4000, 4999)} {rng.randint(1000, 9999)} {rng.randint(1000, 9999)} {i:04d}"
        pan = (
            "".join(rng.choice("ABCDEFGHIJKLMNOPQRSTUVWXYZ") for _ in range(5))
            + f"{i:04d}"
            + rng.choice("ABCDEFGHIJKLMNOPQRSTUVWXYZ")
        )
        aadhaar = f"{rng.randint(1000, 9999)} {rng.randint(1000, 9999)} {i:04d}"
        ifsc = f"HDFC0{i:06d}"
        cust_id = f"CUST-{i:08d}"
        cvv = f"{rng.randint(100, 999)}"
        city = rng.choice(cities)

        record = (
            f"Statement record #{i}: Cardholder Name: {name}. Customer ID: {cust_id}. "
            f"Credit Card Number: {card}. CVV: {cvv}. Mobile Number: {phone}. "
            f"Email: {email}. PAN: {pan}. Aadhaar Number: {aadhaar}. IFSC: {ifsc}. "
            f"Billing City: {city}. Transaction of INR {rng.randint(100, 50000)} was recorded "
            f"on 2026-0{rng.randint(1, 9)}-{rng.randint(10, 28)} for purchase at Merchant {i}."
        )
        records.append(record)
        word_count += len(record.split())
        # ponytail: bare 3-digit CVV values collide by coincidence across a large
        # generated document (e.g. inside unrelated transaction amounts), giving false
        # "leak" hits. Check the labeled phrase ("CVV: 487") instead of the bare digits -
        # that's also what the CVV recognizer regex actually anchors on.
        injected.extend([email, phone, card, pan, aadhaar, ifsc, cust_id, f"CVV: {cvv}"])

    text = "\n".join(records)
    print(f"Generated {pages}-page case: {len(records)} records, {word_count} words, {len(text):,} chars")
    return text, injected


def run_masking_benchmark(text: str, injected_pii: list, label: str):
    """Mask `text`, verify no injected PII leaked, and report time/memory usage."""
    mem_before = psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2) if psutil else None

    tracemalloc.start()
    masker = FinancialPIIMasker(text)
    wall_before = time.perf_counter()
    cpu_before = time.process_time()

    masked = masker.mask_data()

    wall_elapsed = time.perf_counter() - wall_before
    cpu_elapsed = time.process_time() - cpu_before
    _, peak_bytes = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    mem_after = psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2) if psutil else None
    utilisation = (cpu_elapsed / wall_elapsed * 100) if wall_elapsed else 0.0
    leaked = [v for v in injected_pii if v in masked]

    print(f"=== {label} ===")
    print(f"Time to complete : {wall_elapsed:.2f}s wall / {cpu_elapsed:.2f}s CPU ({utilisation:.0f}% utilisation)")
    print(f"Peak traced memory (tracemalloc): {peak_bytes / (1024 ** 2):.1f} MB")
    if mem_before is not None:
        print(f"Process RSS (psutil): {mem_before:.1f} MB -> {mem_after:.1f} MB")
    else:
        print("Process RSS: psutil not installed, skipped")
    print(f"Injected PII values: {len(injected_pii)} | Unmasked leaks: {len(leaked)}")
    if leaked:
        print("LEAKED VALUES (first 10):", leaked[:10])

    assert not leaked, f"{label}: {len(leaked)} PII value(s) leaked through masking!"
    print("PASS - no leaked PII detected.\n")
    return masked


In [ ]:
text_10p, injected_10p = generate_financial_text(10)


In [ ]:
_ = run_masking_benchmark(text_10p, injected_10p, "10-page statement")


In [ ]:
text_20p, injected_20p = generate_financial_text(20)


In [ ]:
_ = run_masking_benchmark(text_20p, injected_20p, "20-page statement")


In [ ]:
text_30p, injected_30p = generate_financial_text(30)


In [ ]:
_ = run_masking_benchmark(text_30p, injected_30p, "30-page statement")


In [ ]:
text_50p, injected_50p = generate_financial_text(50)


In [ ]:
_ = run_masking_benchmark(text_50p, injected_50p, "50-page statement")


In [ ]:
text_100p, injected_100p = generate_financial_text(100)


In [ ]:
_ = run_masking_benchmark(text_100p, injected_100p, "100-page statement")


## Regression test: IVA statement leak fix

Reproduces the exact leak reported from `dummy1.pdf`: debtor name, both address blocks,
and the supervisor name all survived masking under pure NER, and the `IVA...` case
reference numbers weren't covered by any recognizer. Verifies the label-anchored regex
(`_mask_labeled_fields`) and the new `IVA_CASE_REF` pattern close both gaps, while the
field labels themselves stay readable in the output.

In [ ]:
def demo_labeled_fields():
    text = (
        "IVADUMMY7918H\n"
        "Protocol Compliant Annual Report\n"
        "Assignment Details\n"
        "Date Report Issued: 03/10/2024 IP Company Mock Credit \n"
        "Original Creditor Meeting \n"
        "Date: 09/09/2022 IP Case Ref: IVA1dummy18H\n"
        "Debtor: Miss Debtor Angelne Dummy Date of Birth: 05/08/1985\n"
        "Address:\n"
        "Fat A8 Met Set, Whey \n"
        "Brid, HH PLONAK, PO200 7LP, \n"
        "United Kingdom\n"
        "Number of Owned Properties: 0\n"
        "Previous Address: B Maet Set, Diey, \n"
        "STICKORT, SM 999 Date From:\n"
        "Supervisor: Trump Whiter Date Appointed: 26/10/2022\n"
        "Creditors\n"
        "\n"
        "Signature of Supervisor\n"
        "dfgsdhdjfhdk\n"
        "Try Dummy ker\n"
        "\n"
        "IVA1637918H\n"
        "Miss Lin Dummy Name - In Individual Voluntary Arrangement\n"
    )
    m = FinancialPIIMasker(text)
    masked = m.mask_data()

    # values that leaked through in the original bug report must be gone now
    for leaked_value in [
        "IVADUMMY7918H", "IVA1dummy18H", "IVA1637918H",
        "Miss Debtor Angelne Dummy",
        "Fat A8 Met Set", "Whey", "Brid", "HH PLONAK",
        "B Maet Set", "Diey", "STICKORT",
        "Trump Whiter",
        # found by reading the full PDFs (not just the first 500 chars): a signature
        # block with no "Supervisor:" label, and a page-3 title line with no label at all
        "dfgsdhdjfhdk", "Try Dummy ker",
        "Miss Lin Dummy Name",
    ]:
        assert leaked_value not in masked, f"leaked: {leaked_value!r}"

    # the labels themselves must survive so the output stays readable
    for label in ["Debtor:", "Address:", "Previous Address:", "Supervisor:", "Signature of Supervisor"]:
        assert label in masked

    # the fixed suffix phrase must survive too - only the name before it is PII
    assert "In Individual Voluntary Arrangement" in masked

    assert m.original_data() == text

    print("All labeled-field regression checks passed.")
    print(masked)

demo_labeled_fields()


## Performance and Leakage Testing on Large Documents

Below, we define helper functions to generate large text based on our `sample_text` and to capture resource usage (CPU time, system time, peak memory usage, and elapsed execution time). This is a manual/exploratory testing section, run cells here as needed; they don't run automatically with the rest of the notebook.

Note: `resource` is POSIX-only (works in Colab/Linux, not on native Windows).

In [ ]:
import time
import resource


def get_metrics():
    """Utility to retrieve current resource usage."""
    usage = resource.getrusage(resource.RUSAGE_SELF)
    return {
        "time": time.time(),
        "user_cpu": usage.ru_utime,
        "sys_cpu": usage.ru_stime,
        "memory_mb": usage.ru_maxrss / 1024.0,
    }


def print_utilization(start, end):
    """Calculates and prints performance and system utilization metrics."""
    elapsed_time = end["time"] - start["time"]
    user_cpu_used = end["user_cpu"] - start["user_cpu"]
    sys_cpu_used = end["sys_cpu"] - start["sys_cpu"]

    print(f"⏱️  Time to complete: {elapsed_time:.4f} seconds")
    print(f"💻 User CPU Time: {user_cpu_used:.4f} seconds")
    print(f"⚙️  System CPU Time: {sys_cpu_used:.4f} seconds")
    peak_mb = end["memory_mb"]
    print(f"💾 Peak Memory Usage (Instance Total): {peak_mb:.2f} MB")


def check_leakage(text, pii_list=None):
    """Verifies that no original PII has bypassed the masking process."""
    if pii_list is None:
        pii_list = [
            "Rahul Menon",
            "4111 1111 1111 1111",
            "CUST-78451236",
            "HDFC0001234",
            "1234 5678 9012",
            "ABCDE1234F",
            "rahul.menon@example.com",
        ]
    leaked = [item for item in pii_list if item in text]
    if leaked:
        print(f"❌ Leakage Check Failed! Unmasked values found: {leaked}")
    else:
        print("✅ Leakage Check Passed: No original PII items detected in masked text.")


### Test Case: 10 Pages

In [ ]:
# Generate 10 pages worth of text and run masking
text_10_pages = (sample_text + "\n\n") * 10

start_metrics = get_metrics()
masker_10 = FinancialPIIMasker(text_10_pages)
masked_10 = masker_10.mask_data()
end_metrics = get_metrics()


In [ ]:
# Check missed data & system utilization
print("=== 10 Pages Test Results ===")
check_leakage(masked_10)
print_utilization(start_metrics, end_metrics)


### Test Case: 20 Pages

In [ ]:
# Generate 20 pages worth of text and run masking
text_20_pages = (sample_text + "\n\n") * 20

start_metrics = get_metrics()
masker_20 = FinancialPIIMasker(text_20_pages)
masked_20 = masker_20.mask_data()
end_metrics = get_metrics()


In [ ]:
# Check missed data & system utilization
print("=== 20 Pages Test Results ===")
check_leakage(masked_20)
print_utilization(start_metrics, end_metrics)


### Test Case: 30 Pages

In [ ]:
# Generate 30 pages worth of text and run masking
text_30_pages = (sample_text + "\n\n") * 30

start_metrics = get_metrics()
masker_30 = FinancialPIIMasker(text_30_pages)
masked_30 = masker_30.mask_data()
end_metrics = get_metrics()


In [ ]:
# Check missed data & system utilization
print("=== 30 Pages Test Results ===")
check_leakage(masked_30)
print_utilization(start_metrics, end_metrics)


### Test Case: 50 Pages

In [ ]:
# Generate 50 pages worth of text and run masking
text_50_pages = (sample_text + "\n\n") * 50

start_metrics = get_metrics()
masker_50 = FinancialPIIMasker(text_50_pages)
masked_50 = masker_50.mask_data()
end_metrics = get_metrics()


In [ ]:
# Check missed data & system utilization
print("=== 50 Pages Test Results ===")
check_leakage(masked_50)
print_utilization(start_metrics, end_metrics)


### Test Case: 100 Pages

In [ ]:
# Generate 100 pages worth of text and run masking
text_100_pages = (sample_text + "\n\n") * 100

start_metrics = get_metrics()
masker_100 = FinancialPIIMasker(text_100_pages)
masked_100 = masker_100.mask_data()
end_metrics = get_metrics()


In [ ]:
# Check missed data & system utilization
print("=== 100 Pages Test Results ===")
check_leakage(masked_100)
print_utilization(start_metrics, end_metrics)


### Test Case: 1000 Pages

In [ ]:
# Generate 1000 pages worth of text and run masking
text_1000_pages = (sample_text + "\n\n") * 1000

start_metrics = get_metrics()
masker_1000 = FinancialPIIMasker(text_1000_pages)
masked_1000 = masker_1000.mask_data()
end_metrics = get_metrics()


In [ ]:
# Check missed data & system utilization
print("=== 1000 Pages Test Results ===")
check_leakage(masked_1000)
print_utilization(start_metrics, end_metrics)


### PDF-based Testing

Runs the same leakage + performance checks against real PDF files (upload to Colab's `/content/` first, e.g. `dummy1.pdf`, `dummy2.pdf`, `dummy3.pdf`).

In [ ]:
!pip install -q pypdf


In [ ]:
from pypdf import PdfReader


def extract_text_from_pdf(pdf_path: str) -> str:
    """Extracts all text from a given PDF file."""
    try:
        reader = PdfReader(pdf_path)
        text = ""
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
        return text
    except Exception as e:
        print(f"⚠️ Error reading {pdf_path}: {e}")
        return ""


In [ ]:
import os

pdf_files = [
    "/content/dummy1.pdf",
    "/content/dummy2.pdf",
    "/content/dummy3.pdf",
]

# Dictionary to hold our test results for final reporting
pdf_test_results = {}

for pdf_path in pdf_files:
    if not os.path.exists(pdf_path):
        print(f"❌ File {pdf_path} not found.")
        continue

    print(f"\nProcessing {pdf_path}...")
    raw_text = extract_text_from_pdf(pdf_path)
    char_count = len(raw_text)
    word_count = len(raw_text.split())

    print(f"📖 Extracted {char_count} characters, ~{word_count} words.")

    start_metrics = get_metrics()
    masker = FinancialPIIMasker(raw_text)
    masked_text = masker.mask_data()
    end_metrics = get_metrics()

    # Run leakage check + gather metrics
    print(f"=== Results for {os.path.basename(pdf_path)} ===")
    check_leakage(masked_text)
    print_utilization(start_metrics, end_metrics)

    elapsed_time = end_metrics["time"] - start_metrics["time"]
    user_cpu_used = end_metrics["user_cpu"] - start_metrics["user_cpu"]
    sys_cpu_used = end_metrics["sys_cpu"] - start_metrics["sys_cpu"]

    pdf_test_results[os.path.basename(pdf_path)] = {
        "characters": char_count,
        "words": word_count,
        "elapsed_time": elapsed_time,
        "total_cpu": user_cpu_used + sys_cpu_used,
        "peak_memory_mb": end_metrics["memory_mb"],
    }


In [ ]:
# Generate consolidated report of PDF testing
print("=" * 81)
print("                             PDF PII MASKING REPORT                              ")
print("=" * 81)
header = (
    f"{'File Name':<18} | {'Chars':<8} | {'Words':<8} | "
    f"{'Time (s)':<10} | {'CPU (s)':<8} | {'Peak Mem (MB)':<12}"
)
print(header)
print("-" * 81)
for filename, res in pdf_test_results.items():
    row = (
        f"{filename:<18} | {res['characters']:<8} | {res['words']:<8} | "
        f"{res['elapsed_time']:<10.4f} | {res['total_cpu']:<8.4f} | {res['peak_memory_mb']:<12.2f}"
    )
    print(row)
print("=" * 81)


### Visual Verification of Masking Correctness

Side-by-side comparison of the first 500 characters of raw vs. masked text for each tested PDF, so leaks are eyeballable rather than only caught by the substring check above.

In [ ]:
for pdf_path in pdf_files:
    if not os.path.exists(pdf_path):
        continue

    filename = os.path.basename(pdf_path)
    print("=" * 100)
    print(f"📄 VISUAL VERIFICATION FOR: {filename}")
    print("=" * 100)

    raw_text = extract_text_from_pdf(pdf_path)
    masker = FinancialPIIMasker(raw_text)
    masked_text = masker.mask_data()

    print("--- ORIGINAL TEXT SAMPLE (First 500 chars) ---")
    print(raw_text[:500])
    print("\n--- MASKED TEXT SAMPLE (First 500 chars) ---")
    print(masked_text[:500])
    print("\n")


## Additional confirmation tests

Three more checks, each independent of the others:
1. **`dummy2.pdf`-shaped regression** — same IVA template, but this document has no
   `Previous Address` field at all, and the address is in upper case. Confirms the class
   doesn't require every labeled field to be present, and case doesn't matter.
2. **`dummy3.pdf`-shaped regression** — this document puts the address on the *same line*
   as its label (`Address: Cat Bat,...`), and the signature label is glued directly to a
   stray word (`Signature of Supervisor dummyy`) with no space. Confirms the regex isn't
   relying on a specific line layout.
3. **Edge cases** — plain text with no PII at all (must pass through byte-for-byte
   unchanged, i.e. no false-positive redactions), an empty string, and two independent
   instances masked back-to-back (confirms no shared/leaked state between instances).

In [ ]:
def demo_dummy2_regression():
    text = (
        "IVADUMMYY45\n"
        "Protocol Compliant Annual Report\n"
        "Assignment Details\n"
        "Date Report Issued: 29/11/2024 IP Company Mock Company Limited\n"
        "Original Creditor Meeting \n"
        "Date: 28/11/2022 IP Case Ref: IVADUUMY4321\n"
        "Debtor: Mr Seven Dummy Date of Birth: 28/06/1961\n"
        "Address:\n"
        "13 UST Road, \n"
        "South, East Mocking, MCX21 \n"
        "XXX, UNITED KINGDOM\n"
        "Number of Owned Properties: 0\n"
        "Supervisor: Did Dumm Date Appointed: 28/11/2022\n"
        "Creditors\n"
        "\n"
        "Signature of Supervisor\n"
        "David Rankin\n"
        "\n"
        "IVAMOck1987\n"
        "Mr Steven Brady - In Individual Voluntary Arrangement\n"
    )
    m = FinancialPIIMasker(text)
    masked = m.mask_data()

    for leaked_value in [
        "IVADUMMYY45", "IVADUUMY4321", "IVAMOck1987",
        "Mr Seven Dummy",
        "13 UST Road", "South", "East Mocking", "MCX21", "UNITED KINGDOM",
        "Did Dumm",
        "David Rankin",
        "Mr Steven Brady",
    ]:
        assert leaked_value not in masked, f"leaked: {leaked_value!r}"

    # no "Previous Address:" in this document at all - the pattern must not choke on
    # (or accidentally consume text meant for) a field that simply isn't present
    assert "Previous Address" not in masked
    for label in ["Debtor:", "Address:", "Supervisor:", "Signature of Supervisor"]:
        assert label in masked

    assert m.original_data() == text
    print("dummy2-shaped regression checks passed.")

demo_dummy2_regression()


In [ ]:
def demo_dummy3_regression():
    text = (
        "IVA1MOCK45\n"
        "Protocol Compliant Annual Report\n"
        "Assignment Details\n"
        "Date Report Issued: 07/12/2024 IP Company INVALID Limited\n"
        "Original Creditor Meeting \n"
        "Date: 06/12/2022 IP Case Ref: IVA1MOCK451\n"
        "Debtor: Mrs Mocking Minnie Debtor Date of Birth: 18/03/1982\n"
        "Address: Cat Bat,Yale Rad, \n"
        "Bagg, YPL 345 Number of Owned Properties: 0\n"
        "Previous Address: 1 Failedd Avengers, Thee \n"
        "Waffle, U8N DUM Date From: November 2015\n"
        "Supervisor: Dude Rap Date Appointed: 06/12/2022\n"
        "Creditors\n"
        "\n"
        "Signature of Supervisor dummyy\n"
        "Dude Rockyyyy\n"
        "\n"
        "IVADUMMY45\n"
        "Mrs Gladiator Mocking Solitude - In Individual Voluntary Arrangement\n"
    )
    m = FinancialPIIMasker(text)
    masked = m.mask_data()

    for leaked_value in [
        "IVA1MOCK45", "IVA1MOCK451", "IVADUMMY45",
        "Mrs Mocking Minnie Debtor",
        "Cat Bat", "Yale Rad", "Bagg",
        "1 Failedd Avengers", "Thee", "Waffle", "U8N DUM",
        "Dude Rap",
        "dummyy", "Dude Rockyyyy",
        "Mrs Gladiator Mocking Solitude",
    ]:
        assert leaked_value not in masked, f"leaked: {leaked_value!r}"

    for label in ["Debtor:", "Address:", "Previous Address:", "Supervisor:", "Signature of Supervisor"]:
        assert label in masked

    assert m.original_data() == text
    print("dummy3-shaped regression checks passed.")

demo_dummy3_regression()


In [ ]:
def demo_edge_cases():
    # 1. No PII at all: nothing should get redacted (no false positives)
    plain_text = "This is a routine internal memo about office supply budgets and quarterly targets."
    m_plain = FinancialPIIMasker(plain_text)
    assert m_plain.mask_data() == plain_text
    assert m_plain.original_data() == plain_text

    # 2. Empty string in, empty string out, no crash
    m_empty = FinancialPIIMasker("")
    assert m_empty.mask_data() == ""
    assert m_empty.original_data() == ""

    # 3. Two independent instances don't leak state into each other
    text_a = "Email me at alice@example.com about the account."
    text_b = "Email me at bob@example.com about the account."
    masker_a = FinancialPIIMasker(text_a)
    masker_b = FinancialPIIMasker(text_b)

    masked_a = masker_a.mask_data()
    masked_b = masker_b.mask_data()

    assert "alice@example.com" not in masked_a
    assert "bob@example.com" not in masked_b
    assert "bob@example.com" not in masked_a    # instance a never saw instance b's text
    assert "alice@example.com" not in masked_b  # instance b never saw instance a's text
    assert masker_a.original_data() == text_a
    assert masker_b.original_data() == text_b

    print("Edge case checks passed (no PII, empty string, instance isolation).")

demo_edge_cases()
